# Project 4: Initial Alex vs. Kelly classifier

This initial model uses ImageNet-pretrained ResNet-18 as a frozen feature extractor and trains only a two-class output layer. No convolutional blocks are fine-tuned.

Recorded baseline results: **75.7% test accuracy (56/74)** and **79.2% validation accuracy** for the checkpoint selected by minimum validation loss. The original metrics are included in `notebook_outputs/`; the checkpoint remains local. Run the training cells to create it, or copy your existing `best_model.pt` into that folder.

Place the Alex and Kelly image folders beside this notebook, or change `DATA_DIR`. Run cells in order to retrain. Both holdout folders are excluded. The optional last cell only produces holdout predictions.


## 1. Install dependencies
Run this cell once in your notebook environment. Restart the kernel if prompted after installation.


In [ ]:
%pip install torch torchvision Pillow matplotlib


## 2. Imports and settings
Keep the notebook beside the Alex and Kelly folders, or change `DATA_DIR` below.


In [ ]:
import csv
import json
import random
from pathlib import Path

import torch
from PIL import Image
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms


import matplotlib.pyplot as plt
from types import SimpleNamespace

DATA_DIR = Path.cwd()
OUTPUT_DIR = DATA_DIR / "notebook_outputs"
EPOCHS = 20
BATCH_SIZE = 32
SEED = 42
args = SimpleNamespace(data=DATA_DIR, output=OUTPUT_DIR, epochs=EPOCHS,
                       batch_size=BATCH_SIZE, seed=SEED)
CLASSES = ['Alex', 'Kelly']
EXTENSIONS = {'.png', '.jpg', '.jpeg', '.bmp', '.webp'}


## 3. Dataset and evaluation helpers
Images are converted to RGB. Evaluation reports loss, accuracy, and a confusion matrix.


In [ ]:
class Images(Dataset):
    def __init__(self, samples, transform):
        self.samples, self.transform = samples, transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        path, label = self.samples[index]
        with Image.open(path) as image:
            return self.transform(image.convert('RGB')), label


def image_paths(folder):
    return sorted(p for p in folder.rglob('*') if p.suffix.lower() in EXTENSIONS)


def evaluate(model, loader, device):
    model.eval()
    loss_sum, count = 0.0, 0
    confusion = torch.zeros(2, 2, dtype=torch.int64)
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            loss_sum += nn.functional.cross_entropy(logits, labels, reduction='sum').item()
            for actual, predicted in zip(labels.cpu(), logits.argmax(1).cpu()):
                confusion[actual, predicted] += 1
            count += len(labels)
    return loss_sum / count, confusion.diag().sum().item() / count, confusion.tolist()


In [ ]:
random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
weights = models.ResNet18_Weights.DEFAULT
preprocess = weights.transforms()
checkpoint_path = OUTPUT_DIR / 'best_model.pt'
for name in CLASSES:
    count = len(image_paths(DATA_DIR / name))
    print(f'{name}: {count} images')
    if count < 10:
        raise ValueError(f'Check DATA_DIR: expected at least 10 images in {DATA_DIR / name}')
print('Device:', device)


## 4. Preview the images


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for row, name in enumerate(CLASSES):
    for ax, path in zip(axes[row], image_paths(DATA_DIR / name)[:4]):
        with Image.open(path) as image:
            ax.imshow(image.convert('RGB'))
        ax.set_title(name)
        ax.axis('off')
plt.tight_layout()
plt.show()


## 5. Split and preprocess
Split each class into approximately 70% training, 15% validation, and 15% test data. Only training images receive random augmentation. The exact split is saved to CSV.

If images share a source session or contain near-duplicates, split by source group before trusting accuracy; a random image split can give optimistic results.


In [ ]:
# Stratify by class: 70% training, 15% validation, remainder test.
splits = {name: [] for name in ('train', 'validation', 'test')}
rng = random.Random(args.seed)
for label, name in enumerate(CLASSES):
    paths = image_paths(args.data / name)
    if len(paths) < 10:
        raise ValueError(f'Expected at least 10 images in {args.data / name}')
    rng.shuffle(paths)
    a, b = int(len(paths) * .70), int(len(paths) * .85)
    for split, subset in zip(splits, (paths[:a], paths[a:b], paths[b:])):
        splits[split].extend((p, label) for p in subset)
args.output.mkdir(parents=True, exist_ok=True)
with (args.output / 'split.csv').open('w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['path', 'class', 'split'])
    for split, samples in splits.items():
        writer.writerows((str(p.resolve()), CLASSES[label], split) for p, label in samples)
augmentation = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomRotation(8),
    transforms.ToTensor(),
    transforms.Normalize([.485, .456, .406], [.229, .224, .225]),
])
loaders = {name: DataLoader(Images(samples, augmentation if name == 'train' else preprocess),
                           batch_size=args.batch_size, shuffle=name == 'train', num_workers=0)
           for name, samples in splits.items()}

print({name: len(samples) for name, samples in splits.items()})


## 6. Build the model
Freeze the pretrained feature extractor and train a new output layer with two scores: Alex and Kelly.


In [ ]:
model = models.resnet18(weights=weights)
for parameter in model.parameters():
    parameter.requires_grad = False
model.fc = nn.Linear(model.fc.in_features, 2)
model.to(device)
optimizer = torch.optim.Adam(model.fc.parameters(), lr=.001)


## 7. Train
The checkpoint with the lowest validation loss is saved. The test set is reserved for the next step.


In [ ]:
best_loss, history = float('inf'), []
print(f'Device: {device}; split sizes: { {k: len(v) for k, v in splits.items()} }')
for epoch in range(args.epochs):
    # Keep the frozen backbone's batch-normalization statistics fixed.
    model.eval()
    model.fc.train()
    for images, labels in loaders['train']:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = nn.functional.cross_entropy(model(images), labels)
        loss.backward()
        optimizer.step()
    val_loss, val_acc, _ = evaluate(model, loaders['validation'], device)
    history.append({'epoch': epoch + 1, 'validation_loss': val_loss, 'validation_accuracy': val_acc})
    print(f'Epoch {epoch+1:02d}: validation loss={val_loss:.4f}, accuracy={val_acc:.1%}')
    if val_loss < best_loss:
        best_loss = val_loss
        torch.save({'model': model.state_dict(), 'classes': CLASSES, 'seed': args.seed}, checkpoint_path)


## 8. Evaluate on the test set
Load the baseline checkpoint selected by validation loss. Confusion matrix rows are actual labels and columns are predicted labels.


In [ ]:
if 'history' not in globals():
    history = json.loads((OUTPUT_DIR / 'metrics.json').read_text()).get('history', [])
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)
model.load_state_dict(checkpoint['model'])
test_loss, test_acc, confusion = evaluate(model, loaders['test'], device)
metrics = {'classes': CLASSES, 'test_loss': test_loss, 'test_accuracy': test_acc,
           'confusion_matrix_rows_actual_columns_predicted': confusion, 'history': history}
(args.output / 'metrics.json').write_text(json.dumps(metrics, indent=2))
print(f'Test accuracy: {test_acc:.1%}; confusion matrix (Alex, Kelly): {confusion}')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot([h['epoch'] for h in history], [h['validation_loss'] for h in history])
axes[0].set(xlabel='Epoch', ylabel='Loss', title='Validation loss')
axes[1].plot([h['epoch'] for h in history], [h['validation_accuracy'] for h in history])
axes[1].set(xlabel='Epoch', ylabel='Accuracy', title='Validation accuracy', ylim=(0, 1))
axes[2].imshow(confusion, cmap='Blues')
axes[2].set(xticks=[0, 1], yticks=[0, 1], xticklabels=CLASSES, yticklabels=CLASSES,
            xlabel='Predicted', ylabel='Actual', title='Test confusion matrix')
for row in range(2):
    for col in range(2):
        axes[2].text(col, row, confusion[row][col], ha='center', va='center')
plt.tight_layout()
plt.show()


## 9. Predict a new image
Replace the example path with your image. Probabilities are model scores and may not reflect calibrated certainty.


In [ ]:
def predict_image(path):
    model.eval()
    with Image.open(path) as image:
        tensor = preprocess(image.convert('RGB')).unsqueeze(0).to(device)
    with torch.no_grad():
        probabilities = model(tensor).softmax(1)[0].cpu()
    return {'image': str(path), 'prediction': CLASSES[probabilities.argmax().item()],
            'probabilities': dict(zip(CLASSES, probabilities.tolist()))}

# Demonstrate on a reserved test image.
predict_image(splits['test'][0][0])


## 10. Optional: predict holdout images
The holdout folders are unlabeled, so predictions alone cannot measure accuracy.


In [ ]:
# Uncomment after training to save holdout predictions.
# predictions = [predict_image(path) for path in image_paths(DATA_DIR / 'HoldoutSet01')]
# (OUTPUT_DIR / 'holdout_predictions.json').write_text(json.dumps(predictions, indent=2))
